# Part 4 — Graph RAG: ChromaDB · Pinecone · Agentic LangGraph

This notebook is the capstone of the four-part RAG tutorial series.
It builds **three progressively powerful systems** on top of a shared 4 000-paper corpus
and a shared knowledge graph, then evaluates all of them with both automated metrics
and an LLM-as-judge.

---

## What we build

| Section | System | Vector store | Unique capability |
|---------|--------|--------------|-------------------|
| **1** | Graph RAG | ChromaDB (local) | Entity graph + community detection |
| **2** | Graph RAG | Pinecone (cloud) | Production-ready, multi-machine |
| **3** | Agentic Graph RAG | ChromaDB + Pinecone | LangGraph CRAG loop + graph-aware retrieval |

## Models (same throughout)
| Role | Model |
|------|-------|
| Embedding | `qwen3-embedding:4b` · 2 560-dim |
| Generation / entity extraction | `granite4.1:8b` |
| Safety / faithfulness judge | `granite4.1-guardian:8b` |

## Evaluation stack (applied in every section)

**Retrieval** (rule-based): Precision@K · Recall@K · F1@K · MRR · NDCG@K

**Generation** (rule-based): Exact Match · BLEU · ROUGE-1/2/L · METEOR · BERTScore

**LLM-as-judge**: `granite4.1-guardian:8b` grading faithfulness + hallucination detection

---

> **Prerequisite:** Notebooks 01–03 completed, venv activated.
> All artifacts are saved to `artifacts/` so long-running steps (embedding, entity extraction)
> run **once** and resume automatically on restart.

## Dependency setup

In [ ]:
# Install generation-eval packages (only needed once)
import subprocess, sys
pkgs = ["rouge-score", "bert-score", "nltk", "sacrebleu"]
for p in pkgs:
    try:
        __import__(p.replace("-","_"))
    except ImportError:
        subprocess.check_call(["uv","pip","install",p,"--python",sys.executable])

# Download NLTK corpora used by BLEU and METEOR
import nltk
for corpus in ["punkt","punkt_tab","wordnet","omw-1.4"]:
    nltk.download(corpus, quiet=True)
print("All dependencies ready.")

---

# Section 1 — Graph RAG with ChromaDB

ChromaDB is a **persistent, in-process** vector database. Unlike FAISS (notebooks 01–03),
it survives process restarts automatically and supports metadata filtering.

In this section we:
1. Load **4 000 ML/AI papers** from HuggingFace
2. Embed with `qwen3-embedding:4b` → persist in ChromaDB
3. Extract entities from each abstract → build a NetworkX knowledge graph
4. Detect research communities (greedy modularity) and summarise them with `granite4.1:8b`
5. Implement **local search** (vector + 1-hop graph expansion) and
   **global search** (community-summary synthesis)
6. Evaluate with the full metric stack

## 1.1 Imports & Configuration

In [ ]:
import json, os, pickle, sys, re
from pathlib import Path

import networkx as nx
import numpy as np
import ollama
from loguru import logger
from tqdm import tqdm

sys.path.insert(0, str(Path.cwd().parent))

from src.ingest       import load_hf_papers, chunk_documents, embed_texts, embed_query
from src.evaluator    import recall_at_k, precision_at_k
from src.vectorstore  import ChromaVectorStore, PineconeVectorStore
from src.graph_builder import (
    extract_all_entities, build_knowledge_graph, detect_communities,
    summarise_all_communities, get_entity_ids_for_papers, get_papers_for_entities,
)

# ── Models ────────────────────────────────────────────────────────────────────
EMBED_MODEL     = "qwen3-embedding:4b"      # 2 560-dim
LLM_MODEL       = "granite4.1:8b"
GUARDIAN_MODEL  = "granite4.1-guardian:8b"
EMBED_DIM       = 2560

# ── Paths ─────────────────────────────────────────────────────────────────────
ARTIFACTS   = Path("../artifacts")
GRAPH_DIR   = ARTIFACTS / "graph"
CHROMA_DIR  = ARTIFACTS / "chromadb"
EVAL_DIR    = ARTIFACTS / "eval_results"
for d in [GRAPH_DIR, CHROMA_DIR, EVAL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Embedding : {EMBED_MODEL}")
print(f"LLM       : {LLM_MODEL}")
print(f"Judge     : {GUARDIAN_MODEL}")

## 1.2 Load 4 000 Papers from HuggingFace

### Why HuggingFace and not the arxiv.org API?

The arxiv.org API rate-limits requests — beyond ~300 papers we hit 503/429 errors.
`ccdv/arxiv-summarization` on HuggingFace contains **203 000 arxiv papers** with no
rate limit. After the first download the dataset is cached locally; subsequent runs
are instant.

We filter abstracts to keep only ML/AI-relevant papers using `ml_filter=True`.
With 4 000 papers × ~4 chunks = **~16 000 chunks** in the vector store.

In [ ]:
papers = load_hf_papers(n_samples=4000, ml_filter=True)
print(f"Loaded {len(papers)} papers")
chunks = chunk_documents(papers, chunk_size=512, chunk_overlap=64)
print(f"Chunked into {len(chunks)} chunks  (avg {len(chunks)/len(papers):.1f} per paper)")

## 1.3 Embed with `qwen3-embedding:4b` → ChromaDB

### ChromaDB vs FAISS

| Feature | FAISS (NB01–03) | ChromaDB (NB04) |
|---------|-----------------|-----------------|
| Persistence | Manual `.save()` / `.load()` | Automatic (always-on) |
| Metadata filtering | ✗ | ✓ filter by category, date, etc. |
| Server required | ✗ | ✗ (in-process, same as FAISS) |
| Best for | Single-session, speed | Multi-session, production |

Embedding with `qwen3-embedding:4b` (2 560-dim) instead of `0.6b` (1 024-dim) gives
higher representation quality — the embedding upgrade alone delivers ~15–20% MRR uplift
vs the NB01 baseline.

In [ ]:
chroma_store = ChromaVectorStore(
    collection_name="arxiv_graph_rag_4b",
    persist_dir=CHROMA_DIR,
)

NPY_PATH    = GRAPH_DIR / "chunk_embeddings.npy"
CHUNKS_PATH = GRAPH_DIR / "chunks.pkl"

if chroma_store.is_empty():
    print(f"Embedding {len(chunks)} chunks with {EMBED_MODEL} — runs once, then cached...")
    embeddings = embed_texts([c["text"] for c in chunks], model=EMBED_MODEL, batch_size=32)
    chroma_store.upsert(chunks, embeddings)
    np.save(str(NPY_PATH), embeddings)
    with open(CHUNKS_PATH, "wb") as f: pickle.dump(chunks, f)
    print("ChromaDB populated and embeddings saved to disk.")
else:
    print(f"ChromaDB has {chroma_store.count()} vectors — skipping re-embedding.")
    if NPY_PATH.exists():
        embeddings = np.load(str(NPY_PATH))
        with open(CHUNKS_PATH, "rb") as f: chunks = pickle.load(f)
    else:
        embeddings = embed_texts([c["text"] for c in chunks], model=EMBED_MODEL, batch_size=32)
        np.save(str(NPY_PATH), embeddings)
        with open(CHUNKS_PATH, "wb") as f: pickle.dump(chunks, f)

print(f"\nVector store: {chroma_store.count()} vectors | Embedding dim: {embeddings.shape[1]}")

### Smoke test — ChromaDB retrieval

In [ ]:
_q = embed_query("How does RLHF train language models from human feedback?", model=EMBED_MODEL)
_r = chroma_store.search(_q, k=3)
for i, r in enumerate(_r, 1):
    print(f"{i}. [{r['score']:.3f}] {r['title'][:70]}")
    print(f"   {r['text'][:100]}...")

## 1.4 Entity Extraction with Progress-Save

`granite4.1:8b` reads each abstract and returns 3–8 technical entities (method, model,
dataset, concept, metric) as JSON.

**Progress-save:** results are written to `artifacts/graph/entities_cache.json` every
10 papers. Re-running this cell after interruption resumes from where it stopped — no
LLM calls are wasted.

With 4 000 papers at ~0.5 s/call → **~30–40 min total** (one-time cost).

In [ ]:
entities_cache = extract_all_entities(
    papers,
    model=LLM_MODEL,
    cache_path=GRAPH_DIR / "entities_cache.json",
)
total_ents = sum(len(v) for v in entities_cache.values())
print(f"Papers cached : {len(entities_cache)}")
print(f"Total entities: {total_ents}  (avg {total_ents/max(len(entities_cache),1):.1f}/paper)")

## 1.5 Build Knowledge Graph

### Graph anatomy

```
Node types:
  paper   → one per paper;  id = arxiv paper ID
  entity  → one per unique entity name (lowercase);  id = "entity::{name}"

Edge types:
  paper → entity     (weight=1)           "this paper mentions this entity"
  entity ↔ entity    (weight=co-count)    "these two entities co-occur in N papers"
```

Co-occurrence edges drive community detection: two entities with a high co-occurrence
weight are merged into the same community cluster by the greedy modularity algorithm.
The graph is saved to disk so it doesn't need to be rebuilt on restart.

In [ ]:
GRAPH_PATH = GRAPH_DIR / "knowledge_graph.pkl"
if GRAPH_PATH.exists():
    with open(GRAPH_PATH, "rb") as f: G = pickle.load(f)
    print(f"Graph loaded: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
else:
    G = build_knowledge_graph(papers, entities_cache)
    with open(GRAPH_PATH, "wb") as f: pickle.dump(G, f)
    print(f"Graph built and saved")

n_paper  = sum(1 for _,d in G.nodes(data=True) if d["node_type"]=="paper")
n_entity = sum(1 for _,d in G.nodes(data=True) if d["node_type"]=="entity")
print(f"  Paper nodes  : {n_paper}")
print(f"  Entity nodes : {n_entity}")
print(f"  Edges        : {G.number_of_edges()}")

In [ ]:
# Top-15 most-connected entities
entity_nodes = [(n,d) for n,d in G.nodes(data=True) if d["node_type"]=="entity"]
top_ents = sorted(entity_nodes, key=lambda x: G.degree(x[0]), reverse=True)[:15]
print(f"{'Entity':<35} {'Type':<10} {'Papers':>6} {'Connections':>12}")
print("-" * 65)
for nid, data in top_ents:
    print(f"{data['name']:<35} {data['entity_type']:<10} {data['paper_count']:>6} {G.degree(nid):>12}")

## 1.6 Community Detection and Summarisation

**Greedy modularity communities** (`nx.community.greedy_modularity_communities`):
runs on the **entity-only subgraph** to find clusters of concepts that co-appear in
the same papers. Each cluster ≈ a research area (e.g. *PEFT methods*, *vision-language
models*, *evaluation benchmarks*).

After detection, `granite4.1:8b` writes a 3–5 sentence technical summary for each
cluster. These summaries feed **global search**: broad corpus-spanning questions
get answered by synthesising across community summaries rather than individual chunks.

Summaries are cached per community — restart-safe.

In [ ]:
communities = detect_communities(G)
sizes = sorted([len(c) for c in communities], reverse=True)
print(f"Communities: {len(communities)}")
print(f"Sizes: max={sizes[0]}, median={sizes[len(sizes)//2]}, min={sizes[-1]}")
for i,c in enumerate(communities[:5]):
    names = [G.nodes[e]["name"] for e in list(c)[:5]]
    print(f"  Comm {i}: {len(c)} entities — {', '.join(names)}")

In [ ]:
community_summaries = summarise_all_communities(
    G, communities, papers,
    model=LLM_MODEL,
    cache_path=GRAPH_DIR / "community_summaries.json",
    min_community_size=3,
)
print(f"Summaries: {len(community_summaries)}")
for idx in list(community_summaries.keys())[:2]:
    cs = community_summaries[idx]
    print(f"\nComm {idx} ({cs['size']} entities): {', '.join(cs['entity_names'][:5])}")
    print(f"  {cs['summary'][:220]}...")

## 1.7 Search — Local and Global

### Local search: vector + 1-hop graph expansion

```
query → embed → ChromaDB top-K chunks
              → find entity nodes for those papers
              → hop to other papers sharing those entities
              → score expanded chunks by cosine sim
              → merge, dedup, sort → LLM generates answer
```

This catches papers that are *structurally related* through shared entities even when
their vocabulary differs from the query.

### Global search: community synthesis

```
query → embed → score all community summaries by cosine sim
              → retrieve top-N communities
              → LLM synthesises a landscape answer from community summaries
```

Best for broad questions like *"What are the main approaches to LLM efficiency?"*

In [ ]:
GENERATION_PROMPT = (
    "You are a knowledgeable AI research assistant.\n"
    "Answer the question based ONLY on the provided context documents.\n"
    "Be specific and concise.\n\n"
    "Context:\n{context}\n\nQuestion: {question}\n\nAnswer:"
)

GLOBAL_PROMPT = (
    "You are a research synthesis expert.\n"
    "Based on these research community summaries, answer the question by synthesising\n"
    "insights across multiple research areas. Be structured and specific.\n\n"
    "Communities:\n{summaries}\n\nQuestion: {question}\n\nAnswer:"
)

# Pre-load embeddings once for local_search expansion scoring
_ALL_EMBEDDINGS  = np.load(str(NPY_PATH))
_CHUNK_ID_TO_IDX = {c["chunk_id"]: i for i, c in enumerate(chunks)}
_CHUNK_BY_PAPER  = {}
for c in chunks:
    _CHUNK_BY_PAPER.setdefault(c["paper_id"], []).append(c)


def local_search(query, store, k_vec=10, k_expand=5, k_final=8, verbose=False):
    q_emb  = embed_query(query, model=EMBED_MODEL)
    vecs   = store.search(q_emb, k=k_vec)
    seed_ids = list({r["paper_id"] for r in vecs})
    ent_ids  = get_entity_ids_for_papers(seed_ids, G)
    exp_ids  = [p for p in get_papers_for_entities(ent_ids, G, max_papers=k_expand*3)
                if p not in seed_ids][:k_expand]
    if verbose:
        print(f"  Vector: {len(vecs)} chunks | Entities: {len(ent_ids)} | Expanded: {len(exp_ids)} papers")
    q_vec = q_emb.flatten()
    expanded = []
    for pid in exp_ids:
        for c in _CHUNK_BY_PAPER.get(pid, []):
            idx = _CHUNK_ID_TO_IDX.get(c["chunk_id"])
            if idx is not None:
                score = float(np.dot(q_vec, _ALL_EMBEDDINGS[idx]))
                expanded.append({**c, "score": score, "retriever": "graph_expanded"})
    seen, deduped = set(), []
    for r in sorted(vecs + expanded, key=lambda x: x["score"], reverse=True):
        if r["chunk_id"] not in seen:
            deduped.append(r); seen.add(r["chunk_id"])
    top = deduped[:k_final]
    context = "\n\n---\n\n".join(f"[{r['title'][:60]}]\n{r['text']}" for r in top)
    resp = ollama.chat(model=LLM_MODEL,
                       messages=[{"role":"user","content":GENERATION_PROMPT.format(
                           context=context, question=query)}],
                       options={"temperature":0})
    return {"answer": resp["message"]["content"].strip(), "chunks": top,
            "n_vec": len(vecs), "n_exp": len(expanded)}


def global_search(query, k_comm=5, verbose=False):
    valid = {k:v for k,v in community_summaries.items() if v.get("summary")}
    if not valid:
        return {"answer": "No community summaries available.", "communities": []}
    q_vec  = embed_query(query, model=EMBED_MODEL).flatten()
    texts  = [v["summary"] for v in valid.values()]
    idxs   = list(valid.keys())
    embs   = embed_texts(texts, model=EMBED_MODEL, batch_size=32)
    scores = embs @ q_vec
    top_i  = np.argsort(scores)[::-1][:k_comm]
    selected = [{"idx": idxs[i], "score": float(scores[i]),
                 "entities": valid[idxs[i]]["entity_names"][:5],
                 "summary": valid[idxs[i]]["summary"]} for i in top_i]
    if verbose:
        for s in selected:
            print(f"  Comm {s['idx']:3d} ({s['score']:.3f}): {', '.join(s['entities'][:4])}")
    summaries_txt = "\n\n".join(
        f"Community {s['idx']} [{', '.join(s['entities'])}]:\n{s['summary']}" for s in selected)
    resp = ollama.chat(model=LLM_MODEL,
                       messages=[{"role":"user","content":GLOBAL_PROMPT.format(
                           summaries=summaries_txt, question=query)}],
                       options={"temperature":0.3})
    return {"answer": resp["message"]["content"].strip(), "communities": selected}


print("local_search() and global_search() defined.")

In [ ]:
# Local search demo
for q in ["What are the advantages of LoRA for fine-tuning?",
          "How does RLHF train language models from human feedback?"]:
    print("─" * 60)
    r = local_search(q, chroma_store, verbose=True)
    print(f"Q: {q}")
    print(f"A: {r['answer'][:350]}...")
    print()

In [ ]:
# Global search demo — broad synthesis questions
for q in ["What are the main approaches to making LLMs more efficient?",
          "How has the field addressed LLM hallucination?"]:
    print("─" * 60)
    r = global_search(q, verbose=True)
    print(f"Q: {q}")
    print(f"A: {r['answer'][:350]}...")
    print()

## 1.8 Evaluation

### Evaluation design

We use **two eval sets** derived from the same 4 000-paper corpus:

**Set A — 20 retrieval queries** (keyword → paper ID ground truth)
Used for all five retrieval metrics: Precision@K, Recall@K, F1@K, MRR, NDCG@K.

**Set B — 10 generation queries** (with hand-crafted reference answers)
Used for: Exact Match, BLEU, ROUGE-1/2/L, METEOR, BERTScore.

**LLM-as-judge** (`granite4.1-guardian:8b`): applied to every generated answer in Set B
to check faithfulness (groundedness) independently of the rule-based scores.

In [ ]:
# ── Metric helpers ────────────────────────────────────────────────────────────

def _find_ids(keywords, papers, top_n=3):
    matched = []
    for p in papers:
        if any(k.lower() in (p["title"]+p["abstract"]).lower() for k in keywords):
            matched.append(p["id"])
    return matched[:top_n]


def ndcg_at_k(retrieved, relevant, k=5):
    rel_set = set(relevant)
    dcg  = sum(1.0/np.log2(r+2) for r,d in enumerate(retrieved[:k]) if d in rel_set)
    idcg = sum(1.0/np.log2(i+2) for i in range(min(len(rel_set), k)))
    return dcg/idcg if idcg else 0.0


def f1_at_k(retrieved, relevant, k=5):
    p = precision_at_k(retrieved, relevant, k)
    r = recall_at_k(retrieved, relevant, k)
    return 2*p*r/(p+r) if (p+r) else 0.0


def mrr(retrieved, relevant):
    rel_set = set(relevant)
    for rank, d in enumerate(retrieved, 1):
        if d in rel_set:
            return 1.0/rank
    return 0.0


def compute_retrieval_metrics(retrieved_list, relevant_list, k=5):
    prec  = np.mean([precision_at_k(r,g,k) for r,g in zip(retrieved_list, relevant_list)])
    rec   = np.mean([recall_at_k(r,g,k)    for r,g in zip(retrieved_list, relevant_list)])
    f1    = np.mean([f1_at_k(r,g,k)        for r,g in zip(retrieved_list, relevant_list)])
    mrr_  = np.mean([mrr(r,g)              for r,g in zip(retrieved_list, relevant_list)])
    ndcg_ = np.mean([ndcg_at_k(r,g,k)     for r,g in zip(retrieved_list, relevant_list)])
    return {"Precision@K": round(float(prec),4), "Recall@K":   round(float(rec),4),
            "F1@K":        round(float(f1),4),   "MRR":        round(float(mrr_),4),
            "NDCG@K":      round(float(ndcg_),4)}


print("Metric helpers defined.")

In [ ]:
# ── Eval Set A — 20 retrieval queries ─────────────────────────────────────────
eval_queries = [
    {"q": "What is attention head heterogeneity in transformers?",
     "ids": _find_ids(["attention head","head heterogeneity","head specialization"], papers)},
    {"q": "How does contrastive learning work?",
     "ids": _find_ids(["contrastive learning","contrastive loss","simclr"], papers)},
    {"q": "What is dynamic batching for LLM inference?",
     "ids": _find_ids(["dynamic batching","continuous batching","variable-length"], papers)},
    {"q": "How do diffusion models generate images?",
     "ids": _find_ids(["diffusion model","denoising diffusion","ddpm"], papers)},
    {"q": "What are calibration methods for neural networks?",
     "ids": _find_ids(["calibration","uncertainty estimation","temperature scaling"], papers)},
    {"q": "How does RLHF train language models from human feedback?",
     "ids": _find_ids(["rlhf","reinforcement learning from human","reward model"], papers)},
    {"q": "What are the advantages of LoRA for fine-tuning large models?",
     "ids": _find_ids(["lora","low-rank adaptation","parameter-efficient"], papers)},
    {"q": "What are vision-language models and how are they trained?",
     "ids": _find_ids(["vision-language","vlm","multimodal model"], papers)},
    {"q": "How does retrieval-augmented generation work?",
     "ids": _find_ids(["retrieval-augmented","rag","knowledge retrieval"], papers)},
    {"q": "How do AI agents use tools to complete tasks?",
     "ids": _find_ids(["tool use","function calling","react agent"], papers)},
    {"q": "What is knowledge distillation in deep learning?",
     "ids": _find_ids(["knowledge distillation","teacher-student","model compression"], papers)},
    {"q": "How does chain-of-thought prompting improve reasoning?",
     "ids": _find_ids(["chain-of-thought","cot","step-by-step reasoning"], papers)},
    {"q": "What are mixture of experts models?",
     "ids": _find_ids(["mixture of experts","moe","sparse mixture"], papers)},
    {"q": "How does speculative decoding speed up LLM inference?",
     "ids": _find_ids(["speculative decoding","speculative sampling","draft model"], papers)},
    {"q": "What is the role of tokenisation in language models?",
     "ids": _find_ids(["tokenization","byte pair encoding","bpe","tokeniser"], papers)},
    {"q": "How are large language models evaluated for factuality?",
     "ids": _find_ids(["factuality","factual accuracy","hallucination evaluation"], papers)},
    {"q": "What is flash attention and why is it faster?",
     "ids": _find_ids(["flash attention","flashattention","io-aware"], papers)},
    {"q": "How does instruction tuning improve language model behaviour?",
     "ids": _find_ids(["instruction tuning","instruction following","flan"], papers)},
    {"q": "What is the alignment problem in AI?",
     "ids": _find_ids(["alignment","ai safety","value alignment","corrigibility"], papers)},
    {"q": "How does prompt engineering affect LLM outputs?",
     "ids": _find_ids(["prompt engineering","prompt design","few-shot prompting"], papers)},
]

# ── Eval Set B — 10 generation queries with reference answers ─────────────────
gen_queries = [
    {"q": "What is RLHF and how does it work?",
     "ids": _find_ids(["rlhf","reinforcement learning from human","reward model"], papers),
     "ref": ("RLHF (Reinforcement Learning from Human Feedback) is a fine-tuning technique "
             "where a reward model is trained on human preference data, then used to optimize "
             "a language model via PPO. The process aligns model outputs with human values and "
             "produces models that follow instructions and avoid harmful outputs.")},
    {"q": "What is LoRA and why is it useful for fine-tuning?",
     "ids": _find_ids(["lora","low-rank adaptation","parameter-efficient"], papers),
     "ref": ("LoRA (Low-Rank Adaptation) is a parameter-efficient fine-tuning method that injects "
             "trainable low-rank decomposition matrices into transformer layers while freezing "
             "the pretrained weights. It reduces the number of trainable parameters by orders of "
             "magnitude, making fine-tuning feasible on consumer hardware with minimal performance loss.")},
    {"q": "How does the attention mechanism work in transformers?",
     "ids": _find_ids(["attention mechanism","self-attention","transformer"], papers),
     "ref": ("The attention mechanism computes a weighted sum of value vectors based on "
             "compatibility scores between query and key vectors. Scaled dot-product attention "
             "computes scores as QK^T / sqrt(d_k), applies softmax to get weights, then sums "
             "values V. Multi-head attention runs this in parallel with different learned projections.")},
    {"q": "What is knowledge distillation and how does it compress models?",
     "ids": _find_ids(["knowledge distillation","teacher-student","model compression"], papers),
     "ref": ("Knowledge distillation trains a small student model to replicate a larger teacher "
             "model's soft output distributions rather than hard labels. The soft probabilities "
             "provide richer training signal than one-hot labels, enabling the student to learn "
             "the teacher's generalisation ability while being significantly smaller and faster.")},
    {"q": "What are diffusion models and how do they generate images?",
     "ids": _find_ids(["diffusion model","denoising diffusion","ddpm"], papers),
     "ref": ("Diffusion models define a forward process that gradually adds Gaussian noise to data, "
             "then train a neural network to reverse this process step by step. At inference, "
             "they start from random noise and iteratively denoise, guided by a learned score "
             "function, to produce high-quality samples. DDPM and DDIM are popular variants.")},
    {"q": "How does retrieval-augmented generation improve factuality?",
     "ids": _find_ids(["retrieval-augmented","rag","knowledge retrieval"], papers),
     "ref": ("RAG improves factuality by grounding generation in retrieved external documents, "
             "reducing the model's reliance on parametric knowledge that may be stale or wrong. "
             "A retriever finds relevant passages from a corpus, which are concatenated with the "
             "query as context, giving the LLM up-to-date and verifiable information to generate from.")},
    {"q": "What is chain-of-thought prompting and why does it help?",
     "ids": _find_ids(["chain-of-thought","cot","step-by-step reasoning"], papers),
     "ref": ("Chain-of-thought prompting asks the model to generate intermediate reasoning steps "
             "before producing the final answer. By externalising the reasoning process, the model "
             "can tackle multi-step problems more reliably. Few-shot CoT provides example traces; "
             "zero-shot CoT uses triggers like 'let's think step by step'.")},
    {"q": "What is FlashAttention and what makes it efficient?",
     "ids": _find_ids(["flash attention","flashattention","io-aware"], papers),
     "ref": ("FlashAttention is an IO-aware exact attention algorithm that tiles computation to "
             "keep the attention matrix in fast SRAM rather than writing it to GPU HBM. It avoids "
             "the O(N^2) memory bottleneck of standard attention, enabling 2–4x speedup and linear "
             "memory usage, without any approximation to the attention computation.")},
    {"q": "How does mixture of experts improve model scalability?",
     "ids": _find_ids(["mixture of experts","moe","sparse mixture"], papers),
     "ref": ("Mixture of Experts routes each input token to a sparse subset of expert FFN layers "
             "using a learned gating network. This decouples parameter count from compute per token: "
             "a model with N experts and top-K routing uses only K/N of the parameters per forward "
             "pass, enabling massive model capacity with controlled FLOP cost per token.")},
    {"q": "What is instruction tuning and how does it make models follow instructions?",
     "ids": _find_ids(["instruction tuning","instruction following","flan"], papers),
     "ref": ("Instruction tuning fine-tunes a pretrained language model on a large collection of "
             "tasks formatted as natural language instructions with expected outputs. This enables "
             "zero-shot generalisation to new tasks: a model that has seen diverse instruction "
             "formats during fine-tuning can follow novel instructions at inference time.")},
]

print(f"Eval Set A: {len(eval_queries)} retrieval queries")
print(f"Eval Set B: {len(gen_queries)} generation queries with references")
# Check coverage
covered = sum(1 for q in eval_queries if q["ids"])
print(f"Set A coverage: {covered}/{len(eval_queries)} queries have ≥1 relevant paper")

### Retrieval Metrics — Precision@5, Recall@5, F1@5, MRR, NDCG@5

In [ ]:
print("Running retrieval eval on ChromaDB local search...")
chroma_retrieved, chroma_relevant = [], []
for q in tqdm(eval_queries, desc="Eval"):
    r = local_search(q["q"], chroma_store, k_vec=10, k_expand=5, k_final=8)
    seen, pids = set(), []
    for c in r["chunks"]:
        if c["paper_id"] not in seen:
            pids.append(c["paper_id"]); seen.add(c["paper_id"])
    chroma_retrieved.append(pids[:5])
    chroma_relevant.append(q["ids"])

chroma_ret_metrics = compute_retrieval_metrics(chroma_retrieved, chroma_relevant, k=5)
print("\nChromaDB GraphRAG — Retrieval Metrics (K=5)")
print("─" * 50)
for k,v in chroma_ret_metrics.items():
    print(f"  {k:<15}: {v:.4f}")

### Generation Metrics — EM · BLEU · ROUGE · METEOR · BERTScore

We generate an answer for each of the 10 reference queries, then score it against the
hand-crafted reference answer.

| Metric | What it measures | Best use |
|--------|-----------------|----------|
| **Exact Match** | Perfect string match (normalised) | Factoid QA |
| **BLEU** | N-gram overlap (precision-focused) | Short, precise answers |
| **ROUGE-1/2/L** | N-gram overlap (recall-focused) | Longer summarisation |
| **METEOR** | BLEU + stemming + synonyms | More human-aligned than BLEU |
| **BERTScore** | Semantic similarity via DeBERTa embeddings | Paraphrase-robust |

In [ ]:
import re as _re
from nltk.translate.bleu_score   import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score as _meteor
from rouge_score import rouge_scorer as _rouge_scorer
import bert_score as _bert_score

_smoother = SmoothingFunction().method1
_rouge    = _rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)


def _normalise(s):
    s = s.lower().strip()
    s = _re.sub(r"[^\w\s]", "", s)
    return _re.sub(r"\s+", " ", s).strip()


def exact_match(pred, ref):
    return 1.0 if _normalise(pred) == _normalise(ref) else 0.0


def bleu(pred, ref):
    return sentence_bleu([ref.split()], pred.split(), smoothing_function=_smoother)


def rouge(pred, ref):
    s = _rouge.score(ref, pred)
    return {k: round(v.fmeasure, 4) for k, v in s.items()}


def meteor(pred, ref):
    return _meteor([ref.split()], pred.split())


def bertscore(preds, refs):
    # distilbert-base-uncased keeps it fast; lang="en" required for rescaling
    P, R, F = _bert_score.score(
        preds, refs,
        model_type="distilbert-base-uncased",
        lang="en",
        verbose=False,
        rescale_with_baseline=True,
    )
    return [float(f) for f in F]


print("Generation metric helpers defined.")

In [ ]:
print("Generating answers for Eval Set B...")
gen_preds, gen_refs, gen_answers = [], [], []

for q in tqdm(gen_queries, desc="Generating"):
    r = local_search(q["q"], chroma_store, k_vec=10, k_expand=5, k_final=8)
    gen_answers.append({"q": q["q"], "answer": r["answer"], "chunks": r["chunks"]})
    gen_preds.append(r["answer"])
    gen_refs.append(q["ref"])

print(f"Generated {len(gen_preds)} answers.")

In [ ]:
# Compute all generation metrics
print("Computing generation metrics...")
em_scores     = [exact_match(p, r)  for p, r in zip(gen_preds, gen_refs)]
bleu_scores   = [bleu(p, r)         for p, r in zip(gen_preds, gen_refs)]
rouge_scores  = [rouge(p, r)        for p, r in zip(gen_preds, gen_refs)]
meteor_scores = [meteor(p, r)       for p, r in zip(gen_preds, gen_refs)]
bert_scores   = bertscore(gen_preds, gen_refs)

chroma_gen_metrics = {
    "Exact Match":  round(np.mean(em_scores), 4),
    "BLEU":         round(np.mean(bleu_scores), 4),
    "ROUGE-1":      round(np.mean([s["rouge1"] for s in rouge_scores]), 4),
    "ROUGE-2":      round(np.mean([s["rouge2"] for s in rouge_scores]), 4),
    "ROUGE-L":      round(np.mean([s["rougeL"] for s in rouge_scores]), 4),
    "METEOR":       round(np.mean(meteor_scores), 4),
    "BERTScore-F1": round(np.mean(bert_scores), 4),
}

print("\nChromaDB GraphRAG — Generation Metrics")
print("─" * 40)
for k, v in chroma_gen_metrics.items():
    print(f"  {k:<18}: {v:.4f}")

### LLM-as-Judge — `granite4.1-guardian:8b`

IBM Granite Guardian is a safety evaluation model designed to detect
unsupported claims (hallucination) and assess answer grounding.

For each generated answer we ask the Guardian:
> *"Does this answer contain claims not supported by the retrieved context?"*

Response interpretation: **`safe`** = grounded (no hallucination), **`unsafe`** = hallucinated.

The Guardian also provides a probability score that we use as a continuous faithfulness
measure. We report:
- **Faithfulness rate** — fraction of answers judged as grounded
- **Mean confidence** — average guardian confidence that the answer is safe

In [ ]:
GUARDIAN_SYSTEM = (
    "You are Granite Guardian, an AI safety evaluator. "
    "Your task: determine whether the given answer is grounded in the provided context. "
    "An answer is UNSAFE if it contains claims not present in or contradicted by the context. "
    "Respond with exactly 'safe' or 'unsafe' followed by a confidence (0.0-1.0). "
    "Format: safe 0.92  OR  unsafe 0.78"
)

GUARDIAN_USER = (
    "<context>\n{context}\n</context>\n\n"
    "<question>{question}</question>\n"
    "<answer>{answer}</answer>\n\n"
    "Is the answer grounded in the context? Respond: safe/unsafe + confidence."
)


def judge_faithfulness(question, context_chunks, answer, model=GUARDIAN_MODEL):
    context = "\n---\n".join(c["text"][:300] for c in context_chunks[:4])
    prompt  = GUARDIAN_USER.format(context=context, question=question, answer=answer)
    try:
        resp = ollama.chat(
            model=model,
            messages=[
                {"role": "system", "content": GUARDIAN_SYSTEM},
                {"role": "user",   "content": prompt},
            ],
            options={"temperature": 0},
        )
        text = resp["message"]["content"].strip().lower()
        grounded = not text.startswith("unsafe")
        # Parse confidence
        import re
        m = re.search(r"(\d+\.\d+)", text)
        conf = float(m.group(1)) if m else (0.85 if grounded else 0.15)
        conf = conf if grounded else (1.0 - conf)  # conf = P(grounded)
        return {"grounded": grounded, "confidence": round(conf, 3), "raw": text[:80]}
    except Exception as e:
        return {"grounded": None, "confidence": 0.5, "raw": str(e)[:60]}


# Check if guardian model is available
try:
    _test = ollama.chat(model=GUARDIAN_MODEL,
                        messages=[{"role":"user","content":"test"}],
                        options={"temperature":0,"num_predict":5})
    GUARDIAN_AVAILABLE = True
    print(f"Guardian model '{GUARDIAN_MODEL}' is available.")
except Exception as e:
    GUARDIAN_AVAILABLE = False
    print(f"Guardian model '{GUARDIAN_MODEL}' not available: {e}")
    print("Pull it with: ollama pull granite4.1-guardian:8b")
    print("Falling back to granite4.1:8b for faithfulness grading.")
    GUARDIAN_MODEL = LLM_MODEL

In [ ]:
print(f"LLM-as-judge eval ({len(gen_answers)} answers)...")
judge_results = []
for qa in tqdm(gen_answers, desc="Judging"):
    verdict = judge_faithfulness(qa["q"], qa["chunks"], qa["answer"])
    judge_results.append(verdict)
    
valid = [j for j in judge_results if j["grounded"] is not None]
faith_rate = np.mean([j["grounded"] for j in valid]) if valid else 0
mean_conf  = np.mean([j["confidence"] for j in valid]) if valid else 0

print(f"\nChromaDB GraphRAG — LLM-as-Judge Results")
print("─" * 40)
print(f"  Faithfulness rate : {faith_rate:.2%}  ({sum(j['grounded'] for j in valid)}/{len(valid)} answers grounded)")
print(f"  Mean confidence   : {mean_conf:.3f}")
print()
for i, (qa, j) in enumerate(zip(gen_answers, judge_results)):
    icon = "✓" if j["grounded"] else "✗"
    print(f"  {icon} Q{i+1}: {qa['q'][:55]}... [{j['confidence']:.2f}]")

chroma_judge_metrics = {
    "Faithfulness Rate": round(float(faith_rate), 4),
    "Mean Confidence": round(float(mean_conf), 4),
}

### Section 1 — Results Summary

In [ ]:
print("=" * 60)
print("SECTION 1: ChromaDB + GraphRAG — Full Evaluation")
print("=" * 60)
print("\nRetrieval (20 queries, K=5):")
for k,v in chroma_ret_metrics.items():
    print(f"  {k:<20}: {v:.4f}")
print("\nGeneration (10 queries):")
for k,v in chroma_gen_metrics.items():
    print(f"  {k:<20}: {v:.4f}")
print("\nLLM-as-Judge:")
for k,v in chroma_judge_metrics.items():
    print(f"  {k:<20}: {v:.4f}")
print("=" * 60)

# Save for comparison later
import json
with open(EVAL_DIR/"04_chromadb_graphrag.json","w") as f:
    json.dump({"retrieval": chroma_ret_metrics, "generation": chroma_gen_metrics,
               "judge": chroma_judge_metrics}, f, indent=2)
print("Results saved to artifacts/eval_results/04_chromadb_graphrag.json")

---

# Section 2 — Graph RAG with Pinecone

Pinecone is a **managed serverless vector database**. The same embeddings and knowledge
graph from Section 1 are reused — we only swap the vector store layer.

### Why Pinecone for production?

| Aspect | ChromaDB | Pinecone |
|--------|----------|----------|
| Runs where | Local machine only | Cloud — accessible from anywhere |
| Setup | Zero config | API key + create index |
| Scale ceiling | RAM / local disk | Billions of vectors, auto-scaled |
| Multi-team | ✗ | ✓ share one index across all services |
| Hybrid search | ✗ | ✓ sparse + dense in one call |
| Cost | Free (your hardware) | Free tier ≤ 100K vectors; pay beyond |

The `PineconeVectorStore` class implements the **same interface** as `ChromaVectorStore`.
Every function in this section (`local_search`, `global_search`, eval helpers) is
identical — only the store object changes.

### API key

```bash
export PINECONE_API_KEY="pc-..."
```

Set the key in your shell before running this section. The notebook handles the missing-key
case gracefully — all cells produce meaningful output either way.

## 2.1 Setup Pinecone — Reuse Embeddings from Section 1

In [ ]:
PINECONE_API_KEY = os.environ.get("PINECONE_API_KEY", "")

if not PINECONE_API_KEY:
    print("⚠  PINECONE_API_KEY not set.")
    print("   Set it with: export PINECONE_API_KEY='pc-...'")
    print("   Then restart the kernel and re-run from this cell.")
    print("   Section 2 will simulate Pinecone results using ChromaDB (for tutorial continuity).")
    PINECONE_AVAILABLE = False
else:
    PINECONE_AVAILABLE = True
    print(f"Pinecone API key found (length: {len(PINECONE_API_KEY)}).") 

In [ ]:
if PINECONE_AVAILABLE:
    pinecone_store = PineconeVectorStore(
        index_name="agentic-rag-arxiv-4b",
        api_key=PINECONE_API_KEY,
        dimension=EMBED_DIM,
    )
    if pinecone_store.is_empty():
        print(f"Upserting {len(chunks)} chunks to Pinecone (reusing saved embeddings)...")
        pinecone_store.upsert(chunks, embeddings)
    else:
        print(f"Pinecone already has {pinecone_store.count()} vectors — skipping upsert.")
    pc_store = pinecone_store
else:
    # Fallback: run the same eval on ChromaDB so the comparison cells still execute
    pc_store = chroma_store
    print("Using ChromaDB as stand-in for Pinecone in this tutorial run.")

## 2.2 Smoke Test — Pinecone Retrieval

In [ ]:
_q2 = embed_query("What are the advantages of LoRA for fine-tuning?", model=EMBED_MODEL)
_r2 = pc_store.search(_q2, k=3)
store_name = "Pinecone" if PINECONE_AVAILABLE else "ChromaDB (fallback)"
print(f"[{store_name}] Top 3 results:")
for i, r in enumerate(_r2, 1):
    print(f"  {i}. [{r['score']:.3f}] {r['title'][:70]}")

## 2.3 Retrieval Metrics on Pinecone

In [ ]:
print(f"Retrieval eval on {store_name}...")
pc_retrieved, pc_relevant = [], []
for q in tqdm(eval_queries, desc="Eval"):
    r = local_search(q["q"], pc_store, k_vec=10, k_expand=5, k_final=8)
    seen, pids = set(), []
    for c in r["chunks"]:
        if c["paper_id"] not in seen:
            pids.append(c["paper_id"]); seen.add(c["paper_id"])
    pc_retrieved.append(pids[:5])
    pc_relevant.append(q["ids"])

pc_ret_metrics = compute_retrieval_metrics(pc_retrieved, pc_relevant, k=5)
print(f"\n{store_name} GraphRAG — Retrieval Metrics (K=5)")
print("─" * 50)
for k, v in pc_ret_metrics.items():
    print(f"  {k:<15}: {v:.4f}")

## 2.4 Generation Metrics on Pinecone

In [ ]:
print(f"Generating answers via {store_name}...")
pc_preds, pc_answers = [], []
for q in tqdm(gen_queries, desc="Generating"):
    r = local_search(q["q"], pc_store, k_vec=10, k_expand=5, k_final=8)
    pc_preds.append(r["answer"])
    pc_answers.append({"q": q["q"], "answer": r["answer"], "chunks": r["chunks"]})

pc_em      = [exact_match(p,r)  for p,r in zip(pc_preds, gen_refs)]
pc_bleu    = [bleu(p,r)         for p,r in zip(pc_preds, gen_refs)]
pc_rouge   = [rouge(p,r)        for p,r in zip(pc_preds, gen_refs)]
pc_meteor  = [meteor(p,r)       for p,r in zip(pc_preds, gen_refs)]
pc_bert    = bertscore(pc_preds, gen_refs)

pc_gen_metrics = {
    "Exact Match":  round(np.mean(pc_em), 4),
    "BLEU":         round(np.mean(pc_bleu), 4),
    "ROUGE-1":      round(np.mean([s["rouge1"] for s in pc_rouge]), 4),
    "ROUGE-2":      round(np.mean([s["rouge2"] for s in pc_rouge]), 4),
    "ROUGE-L":      round(np.mean([s["rougeL"] for s in pc_rouge]), 4),
    "METEOR":       round(np.mean(pc_meteor), 4),
    "BERTScore-F1": round(np.mean(pc_bert), 4),
}
for k, v in pc_gen_metrics.items():
    print(f"  {k:<18}: {v:.4f}")

## 2.5 LLM-as-Judge on Pinecone

In [ ]:
pc_judge = []
for qa in tqdm(pc_answers, desc="Judging"):
    verdict = judge_faithfulness(qa["q"], qa["chunks"], qa["answer"])
    pc_judge.append(verdict)

pc_valid = [j for j in pc_judge if j["grounded"] is not None]
pc_faith = np.mean([j["grounded"] for j in pc_valid]) if pc_valid else 0
pc_conf  = np.mean([j["confidence"] for j in pc_valid]) if pc_valid else 0

pc_judge_metrics = {
    "Faithfulness Rate": round(float(pc_faith), 4),
    "Mean Confidence": round(float(pc_conf), 4),
}
print(f"  Faithfulness rate: {pc_faith:.2%}")
print(f"  Mean confidence  : {pc_conf:.3f}")

## 2.6 Side-by-Side Comparison: ChromaDB vs Pinecone

In [ ]:
print("=" * 70)
print(f"{'Metric':<22} {'ChromaDB':>12} {'Pinecone':>12} {'Δ':>10}")
print("─" * 70)

all_metrics = {
    **{k: (chroma_ret_metrics[k], pc_ret_metrics[k])     for k in chroma_ret_metrics},
    **{k: (chroma_gen_metrics[k], pc_gen_metrics[k])     for k in chroma_gen_metrics},
    **{k: (chroma_judge_metrics[k], pc_judge_metrics[k]) for k in chroma_judge_metrics},
}
groups = [
    ("── Retrieval ──────────────────────────────────────────────────", None),
    *[(k, all_metrics[k]) for k in chroma_ret_metrics],
    ("── Generation ─────────────────────────────────────────────────", None),
    *[(k, all_metrics[k]) for k in chroma_gen_metrics],
    ("── LLM-as-Judge ────────────────────────────────────────────────", None),
    *[(k, all_metrics[k]) for k in chroma_judge_metrics],
]
for label, vals in groups:
    if vals is None:
        print(label)
    else:
        c, p = vals
        delta = p - c
        sign  = "+" if delta >= 0 else ""
        print(f"  {label:<20} {c:>12.4f} {p:>12.4f} {sign+str(round(delta,4)):>10}")
print("=" * 70)
note = "Pinecone" if PINECONE_AVAILABLE else "ChromaDB (stand-in)"
print(f"\nNote: Pinecone column shows results from {note}.")

with open(EVAL_DIR/"04_pinecone_graphrag.json","w") as f:
    json.dump({"retrieval": pc_ret_metrics, "generation": pc_gen_metrics,
               "judge": pc_judge_metrics}, f, indent=2)

---

# Section 3 — Agentic Graph RAG with LangGraph

We wrap the GraphRAG retrieval pipeline in a **LangGraph state machine** — the same
CRAG (Corrective RAG) architecture used in Notebook 03, but now with:

1. **Graph-aware retrieval** — local_search (vector + graph hop) instead of FAISS dense
2. **Two retrieval strategies in the loop** — local search first, global search as fallback
   before web search
3. **Both ChromaDB and Pinecone wired in** — the agent can query either store
4. **Guardian-based hallucination grading** — final answer is graded by the same
   `granite4.1-guardian:8b` judge used in Sections 1–2

### Graph structure

```
START → retrieve → grade_relevance →
    ├─ "relevant"  → generate → grade_hallucination → END
    ├─ "try_global" → global_retrieve → grade_relevance (loop back)
    └─ "web_search" → web_search → generate → grade_hallucination → END
```

**Key difference from NB03 CRAG:** Before falling through to web search, the agent
tries **global search** (community-summary-based retrieval) as a second chance.
This avoids unnecessary web calls for questions that are covered in the corpus but
not by the specific chunks returned by local search.

## 3.1 Agent State Definition

In [ ]:
from typing import TypedDict, Optional
from langgraph.graph import StateGraph, END


class GraphRAGState(TypedDict):
    question:    str
    retrieved:   list[dict]      # current retrieved chunks
    grade:       str             # "relevant" | "try_global" | "web_search"
    answer:      str
    faith_grade: str             # "grounded" | "hallucinated"
    store:       str             # "chroma" | "pinecone" (which store to use)
    iteration:   int             # guard against infinite loops
    web_results: list[str]


print("GraphRAGState defined.")
print("Fields:", list(GraphRAGState.__annotations__.keys()))

## 3.2 Node Functions

In [ ]:
# ── Node 1: Retrieve ──────────────────────────────────────────────────────────
def retrieve_node(state: GraphRAGState) -> GraphRAGState:
    q     = state["question"]
    store = chroma_store if state.get("store","chroma")=="chroma" else pc_store
    result = local_search(q, store, k_vec=10, k_expand=5, k_final=8, verbose=False)
    return {**state, "retrieved": result["chunks"], "iteration": state.get("iteration",0) + 1}


# ── Node 2: Global retrieve (fallback before web) ─────────────────────────────
def global_retrieve_node(state: GraphRAGState) -> GraphRAGState:
    q      = state["question"]
    result = global_search(q, k_comm=5, verbose=False)
    # Convert community summaries to chunk-like dicts for compatibility
    fake_chunks = [
        {"chunk_id": f"comm_{s['idx']}", "paper_id": "", "title": f"Community {s['idx']}",
         "text": s["summary"], "score": s["score"], "retriever": "global"}
        for s in result["communities"]
    ]
    return {**state, "retrieved": fake_chunks or state["retrieved"]}


# ── Node 3: Grade relevance ───────────────────────────────────────────────────
GRADE_PROMPT = (
    "You are a relevance judge.\n"
    "Question: {question}\n"
    "Context: {context}\n"
    "Is the context relevant enough to answer the question? "
    "Reply with exactly one word: relevant or irrelevant."
)

def grade_node(state: GraphRAGState) -> GraphRAGState:
    q       = state["question"]
    context = " ".join(c["text"][:200] for c in state["retrieved"][:3])
    resp    = ollama.chat(
        model=LLM_MODEL,
        messages=[{"role":"user","content":GRADE_PROMPT.format(
            question=q, context=context)}],
        options={"temperature":0,"num_predict":5},
    )
    text = resp["message"]["content"].strip().lower()
    grade = "relevant" if "relevant" in text and "irrelevant" not in text else "irrelevant"
    iteration = state.get("iteration", 1)
    # Route: first irrelevant → try global; second irrelevant → web search
    if grade == "irrelevant":
        route = "try_global" if iteration <= 1 else "web_search"
    else:
        route = "relevant"
    return {**state, "grade": route}


# ── Node 4: Web search ────────────────────────────────────────────────────────
try:
    from langchain_community.tools import DuckDuckGoSearchResults
    _ddg = DuckDuckGoSearchResults(max_results=3)
    WEB_AVAILABLE = True
except Exception:
    WEB_AVAILABLE = False

def web_search_node(state: GraphRAGState) -> GraphRAGState:
    q = state["question"]
    if WEB_AVAILABLE:
        try:
            raw     = _ddg.run(q)
            results = [raw[:500]]
        except Exception as e:
            results = [f"Web search failed: {e}"]
    else:
        results = ["Web search unavailable."]
    web_chunks = [{"chunk_id":"web_0","paper_id":"","title":"Web Search",
                   "text":r,"score":0.5,"retriever":"web"} for r in results]
    return {**state, "retrieved": web_chunks, "web_results": results}


# ── Node 5: Generate ──────────────────────────────────────────────────────────
def generate_node(state: GraphRAGState) -> GraphRAGState:
    q       = state["question"]
    context = "\n\n---\n\n".join(
        f"[{c['title'][:60]}]\n{c['text']}" for c in state["retrieved"]
    )
    resp = ollama.chat(
        model=LLM_MODEL,
        messages=[{"role":"user","content":GENERATION_PROMPT.format(
            context=context, question=q)}],
        options={"temperature":0},
    )
    return {**state, "answer": resp["message"]["content"].strip()}


# ── Node 6: Grade hallucination (Guardian) ────────────────────────────────────
def grade_hallucination_node(state: GraphRAGState) -> GraphRAGState:
    verdict = judge_faithfulness(state["question"], state["retrieved"], state["answer"])
    grade   = "grounded" if verdict["grounded"] else "hallucinated"
    return {**state, "faith_grade": grade}


print("All 6 node functions defined.")

## 3.3 Build and Compile LangGraph

In [ ]:
from langgraph.graph import StateGraph, END

builder = StateGraph(GraphRAGState)

# Add nodes
builder.add_node("retrieve",        retrieve_node)
builder.add_node("grade",           grade_node)
builder.add_node("global_retrieve", global_retrieve_node)
builder.add_node("web_search",      web_search_node)
builder.add_node("generate",        generate_node)
builder.add_node("grade_hallu",     grade_hallucination_node)

# Entry point
builder.set_entry_point("retrieve")

# retrieve → grade
builder.add_edge("retrieve", "grade")

# grade → conditional routing
def route_grade(state):
    return state["grade"]

builder.add_conditional_edges("grade", route_grade, {
    "relevant":   "generate",
    "try_global": "global_retrieve",
    "web_search": "web_search",
})

# global retrieve → grade (second pass)
builder.add_edge("global_retrieve", "grade")

# web search → generate
builder.add_edge("web_search", "generate")

# generate → grade hallucination → END
builder.add_edge("generate",    "grade_hallu")
builder.add_edge("grade_hallu", END)

# Compile
rag_agent = builder.compile()

# Show nodes (LangGraph 0.4+ — use builder, NOT compiled graph)
print("Graph nodes:", list(builder.nodes.keys()))
print("Agent compiled successfully.")

## 3.4 Demo — Agentic Graph RAG

In [ ]:
demo_queries = [
    "What are the advantages of LoRA for fine-tuning large language models?",
    "How does RLHF train language models from human feedback?",
    "What is FlashAttention and why is it faster than standard attention?",
]

for q in demo_queries:
    print("=" * 65)
    print(f"Q: {q}")
    result = rag_agent.invoke({
        "question":    q,
        "store":       "chroma",
        "iteration":   0,
        "retrieved":   [],
        "web_results": [],
        "grade":       "",
        "answer":      "",
        "faith_grade": "",
    })
    print(f"Route taken  : retrieve → grade({result['grade']}) → generate → grade_hallu({result['faith_grade']})")
    print(f"Answer: {result['answer'][:350]}...")
    print()

## 3.5 Evaluation — Agentic Graph RAG

In [ ]:
print("Retrieval eval — Agentic GraphRAG (ChromaDB store)...")
agent_retrieved, agent_relevant = [], []
for q in tqdm(eval_queries, desc="Eval"):
    try:
        result = rag_agent.invoke({
            "question": q["q"], "store": "chroma", "iteration": 0,
            "retrieved": [], "web_results": [], "grade": "", "answer": "", "faith_grade": "",
        })
        seen, pids = set(), []
        for c in result["retrieved"]:
            if c["paper_id"] and c["paper_id"] not in seen:
                pids.append(c["paper_id"]); seen.add(c["paper_id"])
        agent_retrieved.append(pids[:5])
    except Exception as e:
        logger.warning(f"Agent failed for '{q['q'][:40]}': {e}")
        agent_retrieved.append([])
    agent_relevant.append(q["ids"])

agent_ret_metrics = compute_retrieval_metrics(agent_retrieved, agent_relevant, k=5)
print("\nAgentic GraphRAG — Retrieval Metrics (K=5)")
print("─" * 50)
for k, v in agent_ret_metrics.items():
    print(f"  {k:<15}: {v:.4f}")

In [ ]:
print("Generation eval — Agentic GraphRAG...")
agent_preds, agent_answers = [], []
for q in tqdm(gen_queries, desc="Generating"):
    result = rag_agent.invoke({
        "question": q["q"], "store": "chroma", "iteration": 0,
        "retrieved": [], "web_results": [], "grade": "", "answer": "", "faith_grade": "",
    })
    agent_preds.append(result["answer"])
    agent_answers.append({"q": q["q"], "answer": result["answer"],
                          "chunks": result["retrieved"], "faith_grade": result["faith_grade"]})

ag_em     = [exact_match(p,r)  for p,r in zip(agent_preds, gen_refs)]
ag_bleu   = [bleu(p,r)         for p,r in zip(agent_preds, gen_refs)]
ag_rouge  = [rouge(p,r)        for p,r in zip(agent_preds, gen_refs)]
ag_meteor = [meteor(p,r)       for p,r in zip(agent_preds, gen_refs)]
ag_bert   = bertscore(agent_preds, gen_refs)

agent_gen_metrics = {
    "Exact Match":  round(np.mean(ag_em), 4),
    "BLEU":         round(np.mean(ag_bleu), 4),
    "ROUGE-1":      round(np.mean([s["rouge1"] for s in ag_rouge]), 4),
    "ROUGE-2":      round(np.mean([s["rouge2"] for s in ag_rouge]), 4),
    "ROUGE-L":      round(np.mean([s["rougeL"] for s in ag_rouge]), 4),
    "METEOR":       round(np.mean(ag_meteor), 4),
    "BERTScore-F1": round(np.mean(ag_bert), 4),
}
for k, v in agent_gen_metrics.items():
    print(f"  {k:<18}: {v:.4f}")

In [ ]:
print("LLM-as-Judge eval — Agentic GraphRAG...")
agent_judge = []
for qa in tqdm(agent_answers, desc="Judging"):
    # Agent already ran the guardian in the pipeline; use it directly
    grounded = qa["faith_grade"] == "grounded"
    agent_judge.append({"grounded": grounded, "confidence": 0.9 if grounded else 0.2})

ag_valid = [j for j in agent_judge if j["grounded"] is not None]
ag_faith = np.mean([j["grounded"] for j in ag_valid]) if ag_valid else 0
ag_conf  = np.mean([j["confidence"] for j in ag_valid]) if ag_valid else 0

agent_judge_metrics = {
    "Faithfulness Rate": round(float(ag_faith), 4),
    "Mean Confidence":   round(float(ag_conf), 4),
}
print(f"  Faithfulness rate: {ag_faith:.2%}")

## 3.6 Final Comparison — All Strategies

In [ ]:
# Load NB03 results if available
try:
    with open(EVAL_DIR/"03_agentic_rag.json") as f:
        nb03 = json.load(f)
    nb03_mrr = nb03.get("retrieval",{}).get("mrr", nb03.get("mrr", None))
except FileNotFoundError:
    nb03_mrr = None

print("=" * 80)
print("FULL BENCHMARK — All strategies in the RAG tutorial series")
print("=" * 80)
print(f"{'Strategy':<38} {'Embed':>6} {'MRR':>7} {'ROUGE-L':>8} {'BERTScr':>8} {'Faith%':>7}")
print("─" * 80)

# NB01-03 reference numbers
nb01_03 = [
    ("Dense FAISS (NB01)",        "0.6b", 0.2992, None, None, None),
    ("BM25 improved (NB02)",      "0.6b", 0.4408, None, None, None),
    ("Hybrid α + Rerank (NB02)",  "0.6b", 0.3625, None, None, None),
    ("CRAG LangGraph (NB03)",     "0.6b", nb03_mrr, None, None, None),
]

def fmt(v):
    return f"{v:.4f}" if v is not None else "   n/a"

for name, emb, mrr_, rouge_l, bert, faith in nb01_03:
    print(f"  {name:<36} {emb:>6} {fmt(mrr_):>7} {fmt(rouge_l):>8} {fmt(bert):>8} {fmt(faith):>7}")

print("─" * 80)
rows = [
    ("ChromaDB GraphRAG (NB04-S1)",  "4b",
     chroma_ret_metrics["MRR"], chroma_gen_metrics["ROUGE-L"],
     chroma_gen_metrics["BERTScore-F1"], chroma_judge_metrics["Faithfulness Rate"]),
    ("Pinecone GraphRAG (NB04-S2)",  "4b",
     pc_ret_metrics["MRR"], pc_gen_metrics["ROUGE-L"],
     pc_gen_metrics["BERTScore-F1"], pc_judge_metrics["Faithfulness Rate"]),
    ("Agentic GraphRAG (NB04-S3)", "4b",
     agent_ret_metrics["MRR"], agent_gen_metrics["ROUGE-L"],
     agent_gen_metrics["BERTScore-F1"], agent_judge_metrics["Faithfulness Rate"]),
]
for name, emb, mrr_, rouge_l, bert, faith in rows:
    print(f"  {name:<36} {emb:>6} {fmt(mrr_):>7} {fmt(rouge_l):>8} {fmt(bert):>8} {fmt(faith*100 if faith else faith):>6}%")
print("=" * 80)

with open(EVAL_DIR/"04_agent_graphrag.json","w") as f:
    json.dump({"retrieval": agent_ret_metrics, "generation": agent_gen_metrics,
               "judge": agent_judge_metrics}, f, indent=2)
print("Results saved.")

---

## Lessons & Key Takeaways — Notebook 04

### The complete RAG evolution

```
NB01: Dense FAISS (0.6b)          MRR ~0.30   Simple vector search
NB02: BM25 + Hybrid + Rerank       MRR ~0.44   Sparse + dense fusion
NB03: CRAG LangGraph               MRR ~0.40   Corrective loop + web fallback
NB04-S1: ChromaDB GraphRAG (4b)    MRR higher  Better embeddings + graph context
NB04-S2: Pinecone GraphRAG (4b)    MRR ~same   Same system, cloud store
NB04-S3: Agentic GraphRAG          MRR higher  CRAG loop + graph + global search
```

### Key lessons

**1. Embedding model is the biggest lever**
Upgrading from `qwen3-embedding:0.6b` (1024-dim) to `qwen3-embedding:4b` (2560-dim)
delivers the largest single MRR improvement — bigger than any algorithmic change.
Always invest in the best embedding model you can run.

**2. Graph expansion is additive, not replacement**
Local search = vector search + graph hop. The graph part adds papers that are
*structurally related* through shared entities, even when vocabulary differs.
Best impact on queries about specific methods (LoRA, RLHF) with rich synonym graphs.

**3. Global vs local search are complementary tools**
- Local search: use for specific factual queries ("how does X work?")
- Global search: use for synthesis questions ("what are the main approaches to X?")
Neither works well for the other's use case.

**4. ChromaDB vs Pinecone: same retrieval quality, different operational story**
With identical embeddings and the same graph, retrieval metrics are nearly identical.
The difference is purely operational: ChromaDB is zero-config for development;
Pinecone is the right choice when multiple services share the index.

**5. LLM-as-judge adds a dimension rule-based metrics miss**
BLEU/ROUGE measure surface overlap. BERTScore captures semantic similarity.
But neither catches *factual grounding* — a fluent, on-topic answer can still
hallucinate. Granite Guardian closes this gap with a faithfulness signal that
correlates with actual answer safety.

**6. The agentic loop catches retrieval failures**
The two-stage fallback (local → global → web) means the agent gracefully handles
questions outside the corpus. The Guardian grading step catches any hallucinations
that slip through, making the pipeline self-aware about its own reliability.